<a href="https://colab.research.google.com/github/samhoon000/Real-World-Drug-Surveillance/blob/main/Real_World_Drug_Surveillance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import requests
all_data=[]
for i in range(0,20000,100):
  url ="https://api.fda.gov/drug/event.json?limit=100"
  response=requests.get(url)
  if response.status_code !=200:
    print("Stopped at: ",i)
    break
  data=response.json()
  all_data.extend(data['results'])

In [19]:
import pandas as pd
df_uncleaned=pd.json_normalize(all_data)

In [20]:
df_uncleaned.head()

,safetyreportid,transmissiondateformat,transmissiondate,serious,seriousnessdeath,receivedateformat,receivedate,receiptdateformat,receiptdate,fulfillexpeditecriteria,...,sender.sendertype,receiver.receivertype,receiver.receiverorganization,seriousnessother,occurcountry,patient.patientagegroup,seriousnesshospitalization,patient.summary.narrativeincludeclinical,seriousnesslifethreatening,patient.patientweight
0,5801206-7,102,20090109,1,1,102,20080707,102,20080625,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10003300,102,20141002,1,NaN,102,20140306,102,20140306,2,...,2,6,FDA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10003301,102,20141002,1,NaN,102,20140228,102,20140228,2,...,2,6,FDA,1,NaN,NaN,NaN,NaN,NaN,NaN
3,10003302,102,20141002,2,NaN,102,20140312,102,20140312,2,...,2,6,FDA,NaN,US,NaN,NaN,NaN,NaN,NaN
4,10003304,102,20141212,2,NaN,102,20140312,102,20140424,2,...,2,6,FDA,NaN,US,NaN,NaN,NaN,NaN,NaN


In [21]:
def clean_data(df_uncleaned):
  #Keep the necessary columns
  df=df_uncleaned[[
      'serious',
      'patient.patientonsetage',
      'patient.reaction',
      'patient.drug',
      'occurcountry']].copy()

  # Extract reactions of a patient
  df['reaction_clean']=df['patient.reaction'].apply( lambda lst: [x.get('reactionmeddrapt') for x in lst if 'reactionmeddrapt' in x] if isinstance(lst,list) else [])

  #Extract name of the disease
  df['drug_name']=df['patient.drug'].apply(lambda lst:[x.get('medicinalproduct') for x in lst if 'medicinalproduct' in x ] if isinstance(lst,list) else [])

  #Drop original columns
  df=df.drop(columns=['patient.reaction','patient.drug'])

  #Explode lists → one row per reaction and drug
  df=df.explode('reaction_clean')
  df=df.explode('drug_name')

  #Convert datatype
  df['serious']=pd.to_numeric(df['serious'],errors='coerce')
  df['patient.patientonsetage']=pd.to_numeric(df['patient.patientonsetage'],errors='coerce')

  #Rename columns
  df.rename(columns={'reaction_clean':'reactions','patient.patientonsetage':'patient_age'},inplace=True)

  #Drop Missing values
  df=df.dropna()

  #Reset index values
  df = df.reset_index(drop=True)

  return df

In [22]:
df=clean_data(df_uncleaned)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158600 entries, 0 to 158599
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   serious       158600 non-null  int64  
 1   patient_age   158600 non-null  float64
 2   occurcountry  158600 non-null  object 
 3   reactions     158600 non-null  object 
 4   drug_name     158600 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 6.1+ MB


In [34]:
df.head(10)

,serious,patient_age,occurcountry,reactions,drug_name
0,2,48.0,US,Drug hypersensitivity,LIPITOR
1,2,48.0,US,Drug hypersensitivity,LISINOPRIL
2,2,48.0,US,Drug hypersensitivity,LOSARTAN POTASSIUM
3,2,48.0,US,Drug hypersensitivity,METOPROLOL TARTRATE
4,2,48.0,US,Drug hypersensitivity,MINOCYCLINE
5,2,48.0,US,Drug hypersensitivity,BACTRIM
6,2,68.0,US,Cough,LETAIRIS
7,2,68.0,US,Cough,TYVASO
8,2,68.0,US,Throat irritation,LETAIRIS
9,2,68.0,US,Throat irritation,TYVASO


In [30]:
df.shape

(158600, 5)

# EDA(Exploratory Data Analysis)

In [31]:
# It shows the top 10 most used drug out of this 20000 reports
df['drug_name'].value_counts().head(10)

,count
drug_name,
LETAIRIS,28800
PREDNISONE,6000
RANITIDINE,5400
IDROCLOROTIAZ,5200
IRBESARTAN.,5200
BIBW 2992,5200
OSSICODONE,5200
PARACETAMOLO,5200
CYCLOPHOSPHAMIDE,3000


In [33]:
#Top 10 most common reactions that occur
df['reactions'].value_counts().head(10)

,count
reactions,
Diarrhoea,12000
Pyrexia,9200
Abdominal pain,8800
Blood creatinine increased,8800
C-reactive protein increased,8800
Asthenia,5600
Pleural effusion,4400
Dysuria,4400
Weight decreased,4400


In [37]:
#Shows number of serious pairs (like if a drug and a reaction is serious or not 1 = serious and 2 = not serious)
df['serious'].value_counts()

,count
serious,
1,120800
2,37800


In [39]:
#This shows top reactions which are serious
df[df['serious']==1]['reactions'].value_counts().head(10)

,count
reactions,
Diarrhoea,10800
Pyrexia,9200
C-reactive protein increased,8800
Blood creatinine increased,8800
Abdominal pain,8800
Asthenia,5200
Weight decreased,4400
Dysuria,4400
Pleural effusion,4400
